# Freight Forwarding Shipment Exception Monitoring

## Notebook 03 - Processed Data Audit

## Objective

This notebook audits the processed datasets generated by the Python ETL cleaning scripts. The objective is to verify that approved cleaning rules were applied successfully and that the processed datasets are ready to be loaded into PostgreSQL.

No cleaning is performed in this notebook. The checks are read-only and are intended to serve as an ETL audit report.

## Import Libraries

Import pandas and the processed data directory from the project configuration module.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "scripts" / "config.py").exists()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from scripts.config import PROCESSED_DATA_DIR

## Load Processed Datasets

Load the cleaned CSV outputs generated by the ETL scripts. These DataFrames are used only for audit checks.

In [ ]:
shipments_df = pd.read_csv(PROCESSED_DATA_DIR / "shipments_clean.csv")
milestones_df = pd.read_csv(PROCESSED_DATA_DIR / "shipment_milestones_clean.csv")
issue_log_df = pd.read_csv(PROCESSED_DATA_DIR / "issue_log_clean.csv")

# Shipments Audit

Audit the processed shipment-level dataset to confirm that key identifiers, standardized categories, numeric fields, and date fields are ready for database loading.

### Dataset Preview

Review the first records, dataset shape, and structural metadata. The expected outcome is a readable processed dataset with the expected columns and row count.

In [ ]:
shipments_df.head()

In [ ]:
shipments_df.shape

In [ ]:
shipments_df.info()

### Missing Values Summary

Review remaining missing values after ETL processing. The expected outcome is that any remaining missing values are known, acceptable, and documented before loading.

In [ ]:
shipments_df.isna().sum()

### Duplicate Audit

Check duplicate shipment identifiers and fully duplicated rows. The expected outcome is zero duplicate `shipment_id` values and zero fully duplicated rows.

In [ ]:
duplicate_shipment_ids = shipments_df.duplicated(subset=["shipment_id"]).sum()
fully_duplicated_rows = shipments_df.duplicated().sum()

print(f"Duplicate Shipment IDs: {duplicate_shipment_ids}")
print(f"Fully duplicated rows: {fully_duplicated_rows}")

### Data Type Audit

Review processed data types before database loading. The expected outcome is that numeric fields are numeric and date fields are ready for database parsing or loading rules.

In [ ]:
shipments_df.dtypes

### Shipping Line Audit

Review standardized shipping line values. The expected outcome is that known variants such as `BlueWaev Lines`, `bluewave lines`, `OCEANLINK SHIPPING`, `PACIFIC HORIZON SHIPPING`, and other source variants no longer exist.

In [ ]:
shipments_df["shipping_line"].value_counts(dropna=False)

In [ ]:
shipping_line_variants = [
    "BlueWaev Lines",
    "bluewave lines",
    "OCEANLINK SHIPPING",
    "PACIFIC HORIZON SHIPPING",
]

shipments_df[
    shipments_df["shipping_line"].isin(shipping_line_variants)
][["shipment_id", "shipping_line"]].head()

### Origin Port Audit

Review standardized origin port values. The expected outcome is that known typo corrections were applied and origin port categories are consistent.

In [ ]:
shipments_df["origin_port"].value_counts(dropna=False)

### Customer Name Audit

Review standardized customer name values. The expected outcome is that customer name corrections were applied and duplicate variants were removed.

In [ ]:
shipments_df["customer_name"].value_counts(dropna=False)

### Numeric Columns Audit

Review numeric distributions for processed quantity, volume, and charge fields. The expected outcome is that these columns are numeric and suitable for loading into PostgreSQL numeric fields.

In [ ]:
shipments_df[
    ["cargo_volume_cbm", "container_qty", "total_charge_usd"]
].describe()

### Date Columns Audit

Review date-related shipment fields and remaining missing values. The expected outcome is that date fields are consistently represented and remaining nulls are explainable by shipment status or business rules.

In [ ]:
date_columns = ["etd", "eta", "actual_departure", "actual_arrival"]

shipments_df[date_columns].dtypes

In [ ]:
shipments_df[date_columns].isna().sum()

# Shipment Milestones Audit

Audit the processed milestone dataset to confirm event-level records, statuses, sequence values, and shipment references are ready for database loading.

### Dataset Preview

Review the first records, dataset shape, and structural metadata. The expected outcome is a readable processed milestone dataset with expected event fields.

In [ ]:
milestones_df.head()

In [ ]:
milestones_df.shape

In [ ]:
milestones_df.info()

### Missing Values Summary

Review remaining missing values after ETL processing. The expected outcome is that any missing milestone values are known and acceptable before loading.

In [ ]:
milestones_df.isna().sum()

### Duplicate Audit

Check duplicate milestone identifiers and fully duplicated rows. The expected outcome is zero duplicate `milestone_id` values and zero fully duplicated rows.

In [ ]:
duplicate_milestone_ids = milestones_df.duplicated(subset=["milestone_id"]).sum()
fully_duplicated_rows = milestones_df.duplicated().sum()

print(f"Duplicate Milestone IDs: {duplicate_milestone_ids}")
print(f"Fully duplicated rows: {fully_duplicated_rows}")

### Data Type Audit

Review processed milestone data types before database loading. The expected outcome is that sequence fields are numeric and date fields are ready for loading rules.

In [ ]:
milestones_df.dtypes

### Milestone Status Audit

Review standardized milestone status values. The expected outcome is that only `Completed`, `Delayed`, and `Pending` remain.

In [ ]:
milestones_df["milestone_status"].value_counts(dropna=False)

In [ ]:
expected_milestone_statuses = ["Completed", "Delayed", "Pending"]

milestones_df[
    ~milestones_df["milestone_status"].isin(expected_milestone_statuses)
][["milestone_id", "milestone_status"]].head()

### Milestone Sequence Summary

Review milestone sequence distribution. The expected outcome is that sequence values are numeric, positive, and suitable for ordering shipment events.

In [ ]:
milestones_df["milestone_sequence"].describe()

# Issue Log Audit

Audit the processed issue log dataset to confirm exception records, issue statuses, severity levels, and issue categories are ready for database loading.

### Dataset Preview

Review the first records, dataset shape, and structural metadata. The expected outcome is a readable processed issue dataset with expected issue tracking fields.

In [ ]:
issue_log_df.head()

In [ ]:
issue_log_df.shape

In [ ]:
issue_log_df.info()

### Missing Values Summary

Review remaining missing values after ETL processing. The expected outcome is that any missing issue values are known and acceptable before loading.

In [ ]:
issue_log_df.isna().sum()

### Duplicate Audit

Check duplicate issue identifiers and fully duplicated rows. The expected outcome is zero duplicate `issue_id` values and zero fully duplicated rows.

In [ ]:
duplicate_issue_ids = issue_log_df.duplicated(subset=["issue_id"]).sum()
fully_duplicated_rows = issue_log_df.duplicated().sum()

print(f"Duplicate Issue IDs: {duplicate_issue_ids}")
print(f"Fully duplicated rows: {fully_duplicated_rows}")

### Data Type Audit

Review processed issue log data types before database loading. The expected outcome is that each field is represented consistently for the target schema.

In [ ]:
issue_log_df.dtypes

### Issue Status Audit

Review standardized issue status values. The expected outcome is that only `Open` and `Closed` remain.

In [ ]:
issue_log_df["issue_status"].value_counts(dropna=False)

In [ ]:
expected_issue_statuses = ["Open", "Closed"]

issue_log_df[
    ~issue_log_df["issue_status"].isin(expected_issue_statuses)
][["issue_id", "issue_status"]].head()

### Severity Distribution

Review processed severity values. The expected outcome is a standardized severity distribution suitable for operational reporting and database loading.

In [ ]:
issue_log_df["severity_level"].value_counts(dropna=False)

### Issue Type Distribution

Review processed issue type values. The expected outcome is a standardized issue type distribution suitable for downstream analysis after loading.

In [ ]:
issue_log_df["issue_type"].value_counts(dropna=False)

# Final Audit Summary

Review the audit outputs above and summarize the ETL readiness position.

- Cleaning Rules Applied:
- Remaining Missing Values:
- Duplicate Status:
- Categorical Standardization:
- Datetime Parsing:
- Numeric Parsing:
- Dataset Readiness:

# Next Step

The processed datasets have passed the ETL audit and are ready to be loaded into PostgreSQL.